In [ ]:
%pip install google-generativeai OpenAI pypdf gradio PyPDF2 markdown

In [ ]:
import os
import google.generativeai as genai
from google.generativeai import GenerativeModel
from pypdf import PdfReader
import gradio as gr
from dotenv import load_dotenv
from markdown import markdown



In [ ]:
load_dotenv(override=True)
api_key=os.environ['GOOGLE_API_KEY']
print(f"api_key loaded , starting with: {api_key[:3]}")

genai.configure(api_key=api_key)
model = GenerativeModel("gemini-1.5-flash")

In [ ]:
from bs4 import BeautifulSoup

def prettify_gemini_response(response):
    # Parse HTML
    soup = BeautifulSoup(response, "html.parser")
    # Extract plain text
    plain_text = soup.get_text(separator="\n")
    # Clean up extra newlines
    pretty_text = "\n".join([line.strip() for line in plain_text.split("\n") if line.strip()])
    return pretty_text

# Usage
# pretty_response = prettify_gemini_response(response.text)
# display(pretty_response)


In [ ]:
from PyPDF2 import PdfReader

reader = PdfReader("Profile.pdf")

linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text


In [ ]:
print(linkedin)

In [ ]:
with open("summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [ ]:
name = "Rishabh Dubey"

In [ ]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [ ]:
print(system_prompt)

In [ ]:


# Chat function for Gradio
def chat(message, history):
    # Gemini needs full context manually
    conversation = f"System: {system_prompt}\n"
    for user_msg, bot_msg in history:
        conversation += f"User: {user_msg}\nAssistant: {bot_msg}\n"
    conversation += f"User: {message}\nAssistant:"

    # Create a Gemini model instance
    model = genai.GenerativeModel("gemini-1.5-flash-latest")
    
    # Generate response
    response = model.generate_content([conversation])

    return response.text




In [ ]:
gr.ChatInterface(chat, chatbot=gr.Chatbot()).launch()